# 02 · pandas: ingestion, cleaning and aggregation

**Objective:** To gain insights in using pandas to load, clean, transform, and analyze data from various sources.

## Part 1: Concepts


### 1. Data Ingestion

*Ingestion* means getting data from wherever it lives (a file, a database, a web API) into a pandas **`DataFrame`**: a table in memory, with named columns, one row per record, and an **index** labelling the rows (by default `0, 1, 2, ...`). Every analysis starts here, and whatever the source, the result is the same kind of object, so everything you learn later in this notebook works the same way.

pandas has a family of `pd.read_*` functions, one per format:

| Source | Function |
|---|---|
| CSV / text file | `pd.read_csv` |
| Excel | `pd.read_excel` |
| JSON | `pd.read_json` |
| Parquet (fast binary format for large tables) | `pd.read_parquet` |
| SQL database | `pd.read_sql` |

and the matching `df.to_csv`, `df.to_excel`, ... to write a DataFrame back out.

In this section we load data from the two most common sources: a **CSV file** and an **SQL database**.

**Setup.** The script `database_setup.py` creates a small, synthetic "company" dataset with a fixed random seed, so everyone gets the same numbers. It writes the same two tables (`employees` and `departments`) both as CSV files (`employees.csv`, `departments.csv`) and as an SQLite database (`company.db`). The magic command `%run` executes a Python script from inside the notebook.

In [ ]:
import pandas as pd

# Create company.db, employees.csv and departments.csv in the current folder
%run database_setup.py

**From a CSV file.** A CSV (*comma-separated values*) file is plain text: one line per row, values separated by commas. `pd.read_csv` does a lot of work for us:

* it uses the **first line as the column names**;
* it **guesses the type** of each column (integers, floats, text...) by looking at the values;
* it turns **empty fields** (and strings such as `NaN` or `NA`) into **missing values**, `NaN`.

The file `messy_sales_data.csv` is *deliberately* messy, as real data usually is. Before reading the output, open the file in a text editor and look for: an order that appears twice, empty fields, a price written as `NaN`, and region names with inconsistent capitalisation (`North`, `south`, `EAST`). Section 2 fixes all of these.

Two useful first looks at a new DataFrame are `.head()` (the first 5 rows) and `.info()` (number of rows, column types and non-missing counts).

In [ ]:
df_messy = pd.read_csv('messy_sales_data.csv')
print("--- Messy Sales Data Head ---")
print(df_messy.head())

print("\n--- Column types and missing values ---")
df_messy.info()

**From an SQL database.** A relational database stores data as tables, and you ask it for data with queries written in **SQL** (*Structured Query Language*). Here we use **SQLite**, a database that lives in a single file (`company.db`) and needs no server. Production databases (PostgreSQL, MySQL, ...) work the same way; only the connection string changes.

Reading from a database takes two steps:

1. **Connect.** `create_engine` from the `sqlalchemy` library creates an *engine*: an object that knows how to talk to the database. The string `'sqlite:///company.db'` reads as `<database type>://<location>`: *SQLite*, file `company.db` in the current folder (the third `/` starts a relative path).
2. **Query.** `pd.read_sql(query, engine)` sends the SQL query to the database and returns the result as a DataFrame, with the column names of the table. `SELECT * FROM employees` means "all columns (`*`) of all rows of the table `employees`".

In [ ]:
from sqlalchemy import create_engine

engine = create_engine('sqlite:///company.db')
df_emp = pd.read_sql("SELECT * FROM employees", engine)
print("--- Employees Data Head ---")
print(df_emp.head())
print("Shape:", df_emp.shape)

The point of a database is that it can do part of the work **before** the data reach Python: filtering, sorting and selecting columns happen inside the database, and only the result is transferred. With millions of rows, this matters a lot. For example, the five best paid people in Sales (`department_id = 2`):

In [ ]:
query = '''
SELECT name, salary, tenure_years
FROM employees
WHERE department_id = 2
ORDER BY salary DESC
LIMIT 5
'''
print(pd.read_sql(query, engine))

### 2. Data Cleaning & Preparation
This is the most time-consuming part of data analysis.

**Handling Missing Data (`NaN`)**

In [ ]:
# Check for missing values
print("Missing values before cleaning:")
print(df_messy.isnull().sum())

# Fill numeric NaNs with the column median and ASSIGN the result back.
# Note: since pandas 3.0, df['col'].fillna(value, inplace=True) silently does nothing
# (the column you get with df['col'] behaves as a copy). Always write df['col'] = df['col'].fillna(value).
median_price = df_messy['Price'].median()
df_messy['Price'] = df_messy['Price'].fillna(median_price)

median_qty = df_messy['Quantity'].median()
df_messy['Quantity'] = df_messy['Quantity'].fillna(median_qty)

# Fill categorical NaNs with a placeholder by reassigning the result
df_messy['Region'] = df_messy['Region'].fillna('Unknown')

print("\nMissing values after cleaning:")
print(df_messy.isnull().sum())

print(df_messy)

**Correcting Data Types**

In [ ]:
print("Data types before:", df_messy.dtypes)
df_messy['Quantity'] = df_messy['Quantity'].astype(int)
df_messy['Date'] = pd.to_datetime(df_messy['Date'])
print("\nData types after:", df_messy.dtypes)

**String Manipulation (`.str`)**

In [ ]:
print("Region values before cleanup:", df_messy['Region'].unique())
# Clean whitespace and standardize case
df_messy['Region'] = df_messy['Region'].str.strip().str.title()
print("\nRegion values after cleanup:", df_messy['Region'].unique())

**Handling Duplicates**

In [ ]:
print(f"Shape before dropping duplicates: {df_messy.shape}")
df_messy.drop_duplicates(inplace=True)
print(f"Shape after dropping duplicates: {df_messy.shape}")

### 3. Data Transformation & Aggregation

**Applying Custom Functions (`.apply`)**

In [ ]:
def categorize_price(price):
    return 'High' if price > 200 else 'Low'

df_messy['PriceCategory'] = df_messy['Price'].apply(categorize_price)
print(df_messy[['Price', 'PriceCategory']].head())

**Grouping Data (`.groupby`)**: Essential for summary statistics.

In [ ]:
# Calculate average price per region
avg_price_by_region = df_messy.groupby('Region')['Price'].mean()
print(avg_price_by_region)

**Combining Datasets (`.merge`)**: SQL-style joins.

In [ ]:
df_dept = pd.read_csv('departments.csv')
# Merge employee data with department data
merged_df = pd.merge(df_emp, df_dept, on='department_id', how='left')
print(merged_df.head())

### 4. Basic Statistical Analysis 

**Statistical info of the dataframe (`.describe`)**:

In [ ]:
merged_df.describe()


In [ ]:
merged_df['salary'].describe()

## Part 2: Exercises 

### Lab 2.1: Comprehensive Data Cleanup and Analysis
* **Task:** Perform a full analysis pipeline on the company data.
  1. Load `employees.csv` and `departments.csv`.
  2. Merge the two DataFrames into one.
  3. Create a new column `salary_level` which is 'High' for salaries > 90000 and 'Standard' otherwise.
  4. Use `groupby()` to calculate the number of employees and the average salary for each `department_name`.
  5. Print the final report.

In [ ]:
# Your code here


`pandas-datareader` is listed in `requirements.txt`. If it is missing in your environment, run `%pip install pandas-datareader` in a cell (`%pip` installs into the environment of the running kernel).

### Lab 2.2: Playing around with real data
* **Task:** Perform exploratory analysis on real world economic data.
You will be given the code to use `pandas_reader` to download and clean a public available dataset of economic data.
  1. Explore how a real database is managed ( https://databank.worldbank.org/databases ) .
  2. Explore the numerics of the dataset, using the pandas internal correlation calculation to compute the correlation between gdp and life expectancy, and the correlation between the change of these indicators in time.

In [ ]:
from pandas_datareader import wb

# Define indicators and the time range
indicators = {'NY.GDP.PCAP.KD': 'gdp_per_capita', 'SP.DYN.LE00.IN': 'life_expectancy'}
start_year = 2019
end_year = 2023

# Fetch data from World Bank API
data = wb.download(indicator=list(indicators.keys()), 
                   country='all', 
                   start=start_year, 
                   end=end_year)

# Rename columns for clarity
data = data.rename(columns=indicators)

print("Original Data Sample:")
print(data.head())

In [ ]:
# Reset index to turn 'country' and 'year' into columns
df = data.reset_index()

# Drop rows with any missing values for our indicators
df_clean = df.dropna(subset=['gdp_per_capita', 'life_expectancy'])

# Display info and a sample of the cleaned data
print("\nCleaned Data Info:")
df_clean.info()

print("\nCleaned Data Sample:")
print(df_clean.head())

# Get country metadata to filter out aggregates
countries_info = wb.get_countries()
# Filter for countries only (aggregates such as 'World' or 'Euro area' have region == 'Aggregates')
# Filter our main dataframe to include only countries
df_countries = df_clean[df_clean['country'].isin(countries_info[countries_info['region'] != 'Aggregates']['name'])]


print(f"\nOriginal rows: {len(df_clean)}, Rows after filtering for countries: {len(df_countries)}")


In [ ]:
# Your code here
